# Train YOLO11n for 3D Print Failure Detection

This notebook fine-tunes a YOLO11 nano model on 3D printing failure data and exports it to ONNX for CPU inference.

**Target hardware:** Intel i5-7500T (Lenovo ThinkCentre Tiny / TrueNAS box) — CPU-only inference via ONNX Runtime.

**Model:** YOLO11n (2.6M params) — fastest variant, ~25-50ms per frame on CPU at 640x640.

**Dataset:** [3D Printing Failure Detection](https://universe.roboflow.com/sylucauc/3d-printing-failure-detection) from Roboflow Universe — 5,793 images, 6 classes:
- `blobs`
- `cracks`
- `over_extrusion`
- `spaghetti`
- `stringing`
- `under_extrusion`

**Why upgrade from Obico's YOLOv4?**
- Multi-class detection (6 failure types vs 1)
- Better accuracy (mAP@0.5 ~0.83 vs ~0.60)
- Faster CPU inference (anchor-free architecture)
- Actively maintained (Ultralytics vs abandoned Darknet)

## 1. Setup

Run this on Google Colab with a GPU runtime:
- **Runtime > Change runtime type > T4 GPU**

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
import torch
from ultralytics import YOLO

# Verify GPU is available
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")

## 2. Download Dataset from Roboflow

Get a free API key at [app.roboflow.com](https://app.roboflow.com) (sign up, go to Settings > API Keys).

The dataset has 5,793 annotated images across 6 failure classes.

In [ ]:
from roboflow import Roboflow

# Replace with your Roboflow API key
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # Get from https://app.roboflow.com/settings/api

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("sylucauc").project("3d-printing-failure-detection")
version = project.version(1)
dataset = version.download("yolov8")

print(f"\nDataset downloaded to: {dataset.location}")

In [ ]:
# Inspect the dataset YAML to confirm classes and paths
from pathlib import Path
import yaml

data_yaml_path = Path(dataset.location) / "data.yaml"
with open(data_yaml_path) as f:
    data_config = yaml.safe_load(f)

print("Dataset config:")
print(f"  Classes ({data_config['nc']}): {data_config['names']}")
print(f"  Train: {data_config.get('train', 'N/A')}")
print(f"  Val:   {data_config.get('val', 'N/A')}")
print(f"  Test:  {data_config.get('test', 'N/A')}")

# Count images per split
for split in ["train", "valid", "val", "test"]:
    split_dir = Path(dataset.location) / split / "images"
    if split_dir.exists():
        count = len(list(split_dir.glob("*")))
        print(f"  {split} images: {count}")

## 3. Train YOLO11n

Fine-tune from pretrained COCO weights. Key choices:
- **`yolo11n.pt`**: Nano model (2.6M params) — best for CPU deployment
- **`freeze=10`**: Freeze backbone layers to prevent overfitting on this dataset size
- **`imgsz=640`**: Standard detection resolution
- **`optimizer=AdamW`**: Better convergence on smaller datasets than SGD
- **`lr0=0.001`**: Lower LR for fine-tuning (default 0.01 is for training from scratch)
- **Aggressive augmentation**: Mosaic, mixup, erasing to compensate for dataset size

In [ ]:
# Load pretrained YOLO11 nano
model = YOLO("yolo11n.pt")

# Train
results = model.train(
    # Dataset
    data=f"{dataset.location}/data.yaml",

    # Training duration
    epochs=200,
    patience=50,             # Early stopping: stop if no improvement for 50 epochs

    # Image & batch
    imgsz=640,
    batch=16,                # Reduce to 8 if you get OOM errors

    # Optimizer (AdamW works better than SGD for fine-tuning)
    optimizer="AdamW",
    lr0=0.001,               # Lower LR for fine-tuning from pretrained
    lrf=0.01,                # Final LR = 0.001 * 0.01 = 0.00001
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=5,

    # Transfer learning: freeze backbone layers
    freeze=10,

    # Augmentation (aggressive for this dataset size)
    mosaic=1.0,
    mixup=0.1,
    close_mosaic=15,         # Disable mosaic for last 15 epochs
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,              # No vertical flip — prints are gravity-oriented
    erasing=0.4,

    # Output
    project="3d-print-failure",
    name="yolo11n",
    device=0,
    workers=4,
    amp=True,                # Mixed precision for faster training
    val=True,
)

## 4. Evaluate Results

In [ ]:
# Load the best checkpoint
best_model = YOLO("3d-print-failure/yolo11n/weights/best.pt")

# Run validation
metrics = best_model.val()

print(f"\n{'='*50}")
print(f"Validation Results")
print(f"{'='*50}")
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:     {metrics.box.mp:.4f}")
print(f"Recall:        {metrics.box.mr:.4f}")

In [ ]:
# Per-class breakdown
print(f"\nPer-class AP@0.5:")
print(f"{'-'*40}")
for i, name in enumerate(data_config["names"]):
    class_name = name if isinstance(name, str) else data_config["names"][name]
    ap = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0
    print(f"  {class_name:20s}: {ap:.4f}")

In [ ]:
# Visualize training curves and predictions
from IPython.display import Image, display
from pathlib import Path

results_dir = Path("3d-print-failure/yolo11n")

# Training curves
if (results_dir / "results.png").exists():
    print("Training curves:")
    display(Image(filename=str(results_dir / "results.png"), width=900))

# Confusion matrix
if (results_dir / "confusion_matrix.png").exists():
    print("\nConfusion matrix:")
    display(Image(filename=str(results_dir / "confusion_matrix.png"), width=600))

# Validation predictions
if (results_dir / "val_batch0_pred.jpg").exists():
    print("\nValidation predictions:")
    display(Image(filename=str(results_dir / "val_batch0_pred.jpg"), width=900))

## 5. Test on Your Own Images

Upload your own 3D printer camera images to test before deploying.

In [ ]:
# Test on a sample image from the validation set
import glob

val_images_dir = Path(dataset.location) / "valid" / "images"
if not val_images_dir.exists():
    val_images_dir = Path(dataset.location) / "val" / "images"

sample_images = sorted(glob.glob(str(val_images_dir / "*")))[:6]

if sample_images:
    results = best_model(sample_images, conf=0.25)
    for r in results:
        r.show()
        print(f"  Detections: {len(r.boxes)}")
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            cls_name = r.names[cls_id]
            print(f"    {cls_name}: {conf:.3f}")
        print()

In [ ]:
# Upload and test your own images (run this cell, then use the upload button)
try:
    from google.colab import files
    print("Upload your 3D printer camera images:")
    uploaded = files.upload()

    for filename in uploaded:
        print(f"\n--- {filename} ---")
        results = best_model(filename, conf=0.25)
        results[0].show()
        for box in results[0].boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            cls_name = results[0].names[cls_id]
            print(f"  {cls_name}: {conf:.3f}")
except ImportError:
    print("Not running on Colab. Place images in a local directory and update the path below.")
    # results = best_model("path/to/your/image.jpg", conf=0.25)

## 6. Export to ONNX

Export for CPU inference on the TrueNAS box (i5-7500T).

The ONNX model outputs a single tensor of shape `[1, 4 + num_classes, 8400]`:
- First 4 rows: bounding box (`x_center, y_center, width, height` in pixels)
- Remaining rows: class confidence scores (one per class)
- 8400 candidate detections (from 80x80 + 40x40 + 20x20 feature maps)

In [ ]:
# Export to ONNX
onnx_path = best_model.export(
    format="onnx",
    imgsz=640,
    opset=12,          # Broad compatibility with ONNX Runtime
    simplify=True,     # Optimize graph
    dynamic=False,     # Fixed batch size of 1 (best for edge deployment)
    half=False,        # FP32 for CPU (FP16 is GPU-only)
)

print(f"\nONNX model exported to: {onnx_path}")

In [ ]:
# Verify the ONNX model works with ONNX Runtime (same runtime used in production)
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

print("ONNX model info:")
for inp in session.get_inputs():
    print(f"  Input:  {inp.name} shape={inp.shape} dtype={inp.type}")
for out in session.get_outputs():
    print(f"  Output: {out.name} shape={out.shape} dtype={out.type}")

# Test inference with dummy data
dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)
outputs = session.run(None, {session.get_inputs()[0].name: dummy})
print(f"\nOutput tensor shape: {outputs[0].shape}")
print(f"Expected: [1, {4 + data_config['nc']}, 8400]")

# Quick CPU benchmark
import time
times = []
for _ in range(20):
    start = time.perf_counter()
    session.run(None, {session.get_inputs()[0].name: dummy})
    times.append(time.perf_counter() - start)

print(f"\nColab CPU inference: {np.mean(times)*1000:.1f}ms avg ({np.min(times)*1000:.1f}-{np.max(times)*1000:.1f}ms)")
print(f"Your i5-7500T should be similar or faster.")

## 7. Download Model

Download the trained ONNX model and deploy it to your TrueNAS box.

In [ ]:
import shutil
from pathlib import Path

# Copy ONNX model and class names to a clean output directory
output_dir = Path("model_export")
output_dir.mkdir(exist_ok=True)

# Copy the ONNX file
shutil.copy(onnx_path, output_dir / "model-weights.onnx")

# Write class names file (needed by the detection server)
with open(output_dir / "names", "w") as f:
    names = data_config["names"]
    if isinstance(names, dict):
        for i in sorted(names.keys()):
            f.write(f"{names[i]}\n")
    else:
        for name in names:
            f.write(f"{name}\n")

# Write model metadata
nc = data_config["nc"]
with open(output_dir / "model.meta", "w") as f:
    f.write(f"classes= {nc}\n")
    f.write("names = names\n")
    f.write("model_type = yolo11\n")

# Write model config
with open(output_dir / "model.cfg", "w") as f:
    f.write("[net]\n")
    f.write("batch=1\n")
    f.write("subdivisions=1\n")
    f.write("height=640\n")
    f.write("width=640\n")
    f.write("channels=3\n")

print(f"Model files saved to {output_dir}/")
for f in sorted(output_dir.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.1f} MB")

In [ ]:
# Download files (Colab only)
try:
    from google.colab import files

    # Zip everything for easy download
    shutil.make_archive("yolo11n_3d_print_failure", "zip", output_dir)
    files.download("yolo11n_3d_print_failure.zip")
    print("\nDownload started! After downloading:")
    print("1. Unzip on your TrueNAS box")
    print("2. Place model-weights.onnx in your model cache directory")
    print("3. Update ml_api/model/names with the new class names")
    print("4. Update ml_api/model/model.meta with the new class count")
    print("5. Rebuild: docker compose up -d --build")
except ImportError:
    print(f"Files are in: {output_dir.absolute()}")
    print("Copy model-weights.onnx, names, and model.meta to your ml_api/model/ directory.")

## 8. Deployment

After downloading the model, deploy it to your TrueNAS box:

```bash
# SSH into TrueNAS
ssh truenas

# Navigate to your project
cd /mnt/Main\ Dataset/3d-printer-model

# Pull latest code (with updated detection_model.py that supports YOLO11)
git pull

# Copy your trained model files
# (unzip yolo11n_3d_print_failure.zip first)
cp model-weights.onnx ml_api/model/
cp names ml_api/model/
cp model.meta ml_api/model/
cp model.cfg ml_api/model/

# Rebuild and restart
docker compose down
docker compose up -d --build

# Verify
docker compose logs -f ml_api
# Should show: "YOLO11 ONNX model loaded" and output shape [1, 10, 8400]
```

### Hardware Performance Comparison

| Hardware | PassMark | YOLO11n Inference (est.) | Cost |
|---|---|---|---|
| Your i5-7500T (TrueNAS) | ~5,800 | ~30-50ms | $0 (already own it) |
| Used Tiny PC (i5-6500T) | ~5,500 | ~35-55ms | $50-80 |
| Raspberry Pi 5 | ~1,100 | ~150-250ms | $130 (board+case+PSU+SSD) |
| Raspberry Pi 4 8GB | ~700 | ~300-500ms | $120 |

Your existing TrueNAS box is the clear winner — zero additional cost and inference fast enough for real-time monitoring at 30s intervals.

## Optional: Train YOLO11s (Small) for Better Accuracy

If you want better accuracy and don't mind slightly slower inference (~60-100ms on your i5):

In [ ]:
# Uncomment to train the small variant instead
# model_s = YOLO("yolo11s.pt")
# results_s = model_s.train(
#     data=f"{dataset.location}/data.yaml",
#     epochs=200,
#     patience=50,
#     imgsz=640,
#     batch=16,
#     optimizer="AdamW",
#     lr0=0.001,
#     lrf=0.01,
#     cos_lr=True,
#     freeze=10,
#     mosaic=1.0,
#     mixup=0.1,
#     close_mosaic=15,
#     flipud=0.0,
#     erasing=0.4,
#     project="3d-print-failure",
#     name="yolo11s",
#     device=0,
#     amp=True,
# )
# best_s = YOLO("3d-print-failure/yolo11s/weights/best.pt")
# best_s.export(format="onnx", imgsz=640, opset=12, simplify=True)